In [ ]:
import gymnasium as gym
from gymnasium import spaces
from dataclasses import dataclass

In [ ]:
@dataclass
class WordleConfig:
    vocab: set[str]
    solution: str
    word_length: int = 5
    max_attempts: int = 6
    

    def validate(self):
        self.solution = self.solution.strip().upper()
        assert len(self.solution) == self.word_length, "Solution word must be of length word_length"
        assert self.solution.isalpha(), "Solution word must be alphabetic"
        assert all(isinstance(word, str) for word in self.vocab), "Vocabulary must be a set of strings"
        assert all(word.isalpha() for word in self.vocab), "All vocabulary words must be alphabetic"
        assert all(len(word) == self.word_length for word in self.vocab), "All vocabulary words must have the correct length"

In [ ]:
class WordleEnv(gym.Env):
    def __init__(self, config: WordleConfig):
        super().__init__()
        self.config = config
        self.config.validate()
        self.action_space = spaces.Text(max_length=self.config.word_length)
        self.observation_space = spaces.Dict({
            "greens": spaces.Sequence(spaces.Tuple((spaces.Discrete(26), spaces.Discrete(self.config.word_length)))),
            "yellows": spaces.Sequence(spaces.Tuple((spaces.Discrete(26), spaces.Discrete(self.config.word_length)))),
            "grays": spaces.Sequence(spaces.Discrete(26)),
            "num_attempts": spaces.Discrete(self.config.max_attempts + 1),
            "done": spaces.Discrete(2)
        })
        self._is_done = False
        self.reset()
    
    def reset(self):
        super().reset()
        self.attempts = 0
        self.done = False
        self.greens = []
        self.yellows = []
        self.grays = set()
        self.info = []
        return self._get_obs(), self.info
    
    def step(self, action):
        assert not self.done, "Episode is done. Please reset the environment."
        action = action.strip().upper()
        assert len(action) == self.config.word_length, f"Action must be a word of length {self.config.word_length}"
        assert action.isalpha(), "Action must be an alphabetic word"
        assert action in self.config.vocab, "Action must be a valid word in the vocabulary"
        
        self.attempts += 1
        feedback = self._score_guess(action, self.config.solution)
        self.info.append((action, feedback))
        
        if action == self.config.solution:
            reward = 1.0
            self.done = True
        elif self.attempts >= self.config.max_attempts:
            reward = -1.0
            self.done = True
        else:
            reward = 0.0
        
        return self._get_obs(), reward, self.done, False, self.info
    
    def _get_obs(self):
        return {
            "greens": self.greens,
            "yellows": self.yellows,
            "grays": list(self.grays),
            "num_attempts": self.attempts,
            "done": self.done
        }
    
    def _update_knowledge_from_guess(self, guess, feedback):
        for i, (g_char, f) in enumerate(zip(guess, feedback)):
            if f == "green":
                self.greens.append((ord(g_char) - ord('A'), i))
            elif f == "yellow":
                self.yellows.append((ord(g_char) - ord('A'), i))
            elif f == "gray":
                self.grays.add(ord(g_char) - ord('A'))
    
    @staticmethod
    def _score_guess(guess, solution):
        feedback = []
        for g_char, s_char in zip(guess, solution):
            if g_char == s_char:
                feedback.append("green")
            elif g_char in solution:
                feedback.append("yellow")
            else:
                feedback.append("gray")
        return feedback

In [ ]:
solution = "CRANE"
valid_words = {"CRANE", "SLATE", "TRACE", "REACT", "CRATE", "STONE", "ARISE", "RAISE", "LEAST"}

config = WordleConfig(solution=solution, vocab=valid_words, word_length=5, max_attempts=6)
env = WordleEnv(config)

obs, info = env.reset()
while not env._is_done:
    guess = input("Enter your guess: ").strip().upper()
    obs, reward, terminated, truncated, info = env.step(guess)

    print(f"Reward: {reward}")
    print(f"Attempt: {obs['num_attempts']}/{config.max_attempts}")
    print(obs)
    for g, f in obs["info"]:
        print(f"{g}  ->  {f}")
    print()

    if terminated:
        if info["history"] and info["history"][-1][0] == config.solution:
            print("🎉 You got it!")
        else:
            print(f"Game Over. Solution was: {config.solution}")
        break

Reward: 0.0
Attempt: 1/6
{'greens': [], 'yellows': [], 'grays': [], 'num_attempts': 1, 'done': False}


KeyError: 'info'